# Wine Quality Classification with Hyperparameter Tuning
## Data Mining – Classification & Model Tuning
### Classifiers: Logistic Regression & Support Vector Machine (SVM)
**Dataset:** Red Wine Quality (UCI ML Repository)  
**Author:** Faus7679  


## 1. Load Software Packages

In [ ]:
# Core data libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn – preprocessing & feature engineering
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

# Scikit-learn – models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Scikit-learn – model selection & evaluation
from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score,
    GridSearchCV, learning_curve
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score,
    f1_score, roc_curve, auc, accuracy_score
)

# Class imbalance
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

print("All packages loaded successfully.")


✅ All packages loaded successfully.


## 2. Load & Explore Data

In [2]:
import os

# Load dataset – supports semicolon-delimited (UCI) or comma-delimited formats
data_paths = [
    'wine-quality-red.csv',
    os.path.expanduser('~/Downloads/wine-quality-red.csv'),
]

df = None
for path in data_paths:
    if os.path.exists(path):
        # Try semicolon first (UCI format), then comma
        try:
            tmp = pd.read_csv(path, sep=';')
            if tmp.shape[1] >= 12:
                df = tmp
                print(f"Loaded (semicolon-delimited): {path}")
            else:
                df = pd.read_csv(path)
                print(f"Loaded (comma-delimited): {path}")
        except Exception:
            df = pd.read_csv(path)
            print(f"Loaded: {path}")
        break

if df is None:
    raise FileNotFoundError(
        "wine-quality-red.csv not found. "
        "Place the file in the repository root or in ~/Downloads/"
    )

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


Loaded (semicolon-delimited): wine-quality-red.csv

Dataset shape: (1599, 12)
Columns: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,9.2,0.40,0.20,2.4,0.213,5,8,0.9951,3.34,0.64,8.9,6
1,8.1,0.50,0.36,1.9,0.017,10,27,0.9929,3.26,0.80,11.3,6
2,9.4,0.77,0.31,2.1,0.015,1,6,1.0010,3.42,0.53,9.1,5
3,11.0,0.41,0.35,6.3,0.178,2,6,0.9973,3.28,0.71,9.5,6
4,7.9,0.38,0.31,1.9,0.057,7,23,0.9963,3.64,0.68,10.7,6


In [3]:
# Descriptive statistics
print("=== Descriptive Statistics ===")
print(df.describe().to_string())


=== Descriptive Statistics ===
       fixed acidity  volatile acidity  citric acid  residual sugar    chlorides  free sulfur dioxide  total sulfur dioxide      density           pH    sulphates      alcohol      quality
count    1599.000000       1599.000000  1599.000000     1599.000000  1599.000000          1599.000000           1599.000000  1599.000000  1599.000000  1599.000000  1599.000000  1599.000000
mean        8.399187          0.529031     0.274622        2.651345     0.068370            12.833646             38.534084     0.996672     3.313496     0.662545    10.438399     5.636023
std         1.705529          0.174425     0.179972        1.455157     0.055163            11.401363             35.703601     0.001916     0.155729     0.165382     1.059572     0.807569
min         4.600000          0.120000     0.000000        1.200000     0.012000             1.000000              6.000000     0.990000     2.740000     0.330000     8.400000     3.000000
25%         7.200000    

In [4]:
# Quality distribution
print("=== Quality Score Distribution ===")
print(df['quality'].value_counts().sort_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart of quality
df['quality'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Wine Quality Score Distribution')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Correlation heatmap
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1], linewidths=0.5, annot_kws={'size': 7})
axes[1].set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved → eda_plots.png")


=== Quality Score Distribution ===
quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64


Plot saved → eda_plots.png


## 3. Preprocessing

### a) Missing Data Technique
We apply **median imputation** via `SimpleImputer`. Median is robust to outliers, which is important for skewed chemical measurements like residual sugar and chlorides.

### b) Feature Engineering (two techniques)
1. **Log transformation** – normalises right-skewed features (residual sugar, chlorides, sulfur dioxide, sulphates).
2. **Interaction / derived features** – captures non-linear relationships (alcohol × sulphates, total acidity sum, volatile-to-citric ratio).

### c) Feature Selection
**SelectKBest** with the ANOVA F-test ranks features by their univariate relationship with the target, retaining the 10 most informative features.


In [5]:
# ── a) Missing data: median imputation ──────────────────────────────────
print("=== a) Missing Data Handling ===")
print(f"Missing values BEFORE imputation: {df.isnull().sum().sum()}")

imputer = SimpleImputer(strategy='median')
df_imp = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print(f"Missing values AFTER  imputation: {df_imp.isnull().sum().sum()}")


=== a) Missing Data Handling ===
Missing values BEFORE imputation: 0
Missing values AFTER  imputation: 0


In [6]:
# ── b) Feature Engineering ───────────────────────────────────────────────
print("\n=== b) Feature Engineering ===")

df_eng = df_imp.copy()

# Technique 1: log1p transformation of right-skewed features
skewed = ['residual sugar', 'chlorides',
          'free sulfur dioxide', 'total sulfur dioxide', 'sulphates']
for feat in skewed:
    df_eng[f'log_{feat.replace(" ", "_")}'] = np.log1p(df_imp[feat])
print(f"Technique 1 – log-transformed {len(skewed)} skewed features.")

# Technique 2: domain-driven interaction features
df_eng['alcohol_x_sulphates']  = df_eng['alcohol'] * df_eng['sulphates']
df_eng['volatile_to_citric']   = (df_eng['volatile acidity'] /
                                   (df_eng['citric acid'] + 1e-3))
df_eng['total_acidity']        = (df_eng['fixed acidity'] +
                                   df_eng['volatile acidity'] +
                                   df_eng['citric acid'])
print("Technique 2 – added interaction features: "
      "alcohol_x_sulphates, volatile_to_citric, total_acidity")
print(f"Total features after engineering: {df_eng.shape[1]}")



=== b) Feature Engineering ===
Technique 1 – log-transformed 5 skewed features.
Technique 2 – added interaction features: alcohol_x_sulphates, volatile_to_citric, total_acidity
Total features after engineering: 20


In [7]:
# ── Binary target: Good (quality ≥ 6) vs Poor (quality < 6) ─────────────
df_eng['quality_binary'] = (df_eng['quality'] >= 6).astype(int)

X_all = df_eng.drop(columns=['quality', 'quality_binary'])
y     = df_eng['quality_binary']

print("Binary target class distribution:")
print(y.value_counts().rename({0: 'Poor (0)', 1: 'Good (1)'}))
print(f"Class ratio (Good/Poor): {y.mean():.2%}")


Binary target class distribution:
quality_binary
Good (1)    855
Poor (0)    744
Name: count, dtype: int64
Class ratio (Good/Poor): 53.47%


In [8]:
# ── c) Feature Selection: SelectKBest (ANOVA F-test) ─────────────────────
print("\n=== c) Feature Selection ===")

scaler_fs = StandardScaler()
X_scaled_all = scaler_fs.fit_transform(X_all)

selector = SelectKBest(score_func=f_classif, k=10)
selector.fit(X_scaled_all, y)

feat_scores = pd.DataFrame({
    'Feature': X_all.columns,
    'F-Score': selector.scores_,
    'P-value': selector.pvalues_
}).sort_values('F-Score', ascending=False)

print("\nFeature Ranking (top 10 selected):")
print(feat_scores.to_string(index=False))

selected_features = X_all.columns[selector.get_support()].tolist()
print(f"\nSelected features: {selected_features}")

# Plot feature importance
plt.figure(figsize=(10, 5))
top15 = feat_scores.head(15).sort_values('F-Score')
colors = ['teal' if f in selected_features else 'lightgray'
          for f in top15['Feature']]
plt.barh(top15['Feature'], top15['F-Score'], color=colors)
plt.xlabel('ANOVA F-Score')
plt.title('Feature Importance (SelectKBest – ANOVA F-test)\n'
          'Teal = selected, Gray = excluded')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved → feature_importance.png")



=== c) Feature Selection ===

Feature Ranking (top 10 selected):
                 Feature      F-Score       P-value
        volatile acidity 2.082048e+03 1.053929e-291
     alcohol_x_sulphates 7.199546e+01  4.847286e-17
                 alcohol 4.756057e+01  7.657519e-12
               sulphates 4.329823e+01  6.354114e-11
           log_sulphates 4.304933e+01  7.191950e-11
      volatile_to_citric 4.929349e+00  2.654419e-02
           total_acidity 2.884301e+00  8.964160e-02
           fixed acidity 1.861100e+00  1.726901e-01
    total sulfur dioxide 6.234610e-01  4.298813e-01
                 density 5.637916e-01  4.528472e-01
log_total_sulfur_dioxide 5.565549e-01  4.557615e-01
               chlorides 4.632802e-01  4.961936e-01
 log_free_sulfur_dioxide 3.964359e-01  5.290257e-01
           log_chlorides 3.856725e-01  5.346716e-01
      log_residual_sugar 2.063622e-01  6.496955e-01
     free sulfur dioxide 6.780546e-02  7.945928e-01
                      pH 3.144645e-02  8.592705e-0

Plot saved → feature_importance.png


## 4. Subset the Data & Split into Training / Testing Sets

In [9]:
# ── Subset: use all 1 599 instances (small dataset; no further subsetting needed) ──
X = selector.transform(X_scaled_all)
print(f"Final feature matrix: {X.shape}")
print(f"Class distribution  : {pd.Series(y.values).value_counts().to_dict()}")


Final feature matrix: (1599, 10)
Class distribution  : {1: 855, 0: 744}


In [10]:
# ── Stratified 80 / 20 split ────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape}  "
      f"| class dist → {pd.Series(y_train).value_counts().to_dict()}")
print(f"Test set     : {X_test.shape}  "
      f"| class dist → {pd.Series(y_test).value_counts().to_dict()}")


Training set : (1279, 10)  | class dist → {1: 684, 0: 595}
Test set     : (320, 10)  | class dist → {1: 171, 0: 149}


## 5 & 6. Classifier 1 – Logistic Regression

### c) Class Imbalance: SMOTE
Synthetic Minority Over-sampling Technique (SMOTE) is applied **only to the training set** to generate synthetic samples for the minority class, avoiding information leakage into the test set.


In [11]:
# ── c) SMOTE on training data ────────────────────────────────────────────
smote = SMOTE(random_state=42)
X_tr_sm, y_tr_sm = smote.fit_resample(X_train, y_train)

print("Class distribution BEFORE SMOTE:", pd.Series(y_train).value_counts().to_dict())
print("Class distribution AFTER  SMOTE:", pd.Series(y_tr_sm).value_counts().to_dict())


Class distribution BEFORE SMOTE: {1: 684, 0: 595}
Class distribution AFTER  SMOTE: {1: 684, 0: 684}


In [12]:
# ── a) K-Fold Cross-Validation (k = 5) ───────────────────────────────────
print("=== a) 5-Fold Cross-Validation – Logistic Regression ===")

lr_base = LogisticRegression(random_state=42, max_iter=1000)
kfold   = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_lr = cross_val_score(lr_base, X_tr_sm, y_tr_sm,
                                cv=kfold, scoring='accuracy')
print(f"CV fold scores : {np.round(cv_scores_lr, 4)}")
print(f"Mean accuracy  : {cv_scores_lr.mean():.4f}  "
      f"(±{cv_scores_lr.std()*2:.4f})")


=== a) 5-Fold Cross-Validation – Logistic Regression ===
CV fold scores : [0.9453 0.9416 0.9562 0.9231 0.9634]
Mean accuracy  : 0.9459  (±0.0276)


In [13]:
# ── b) Grid Search Hyperparameter Tuning ─────────────────────────────────
print("=== b) Grid Search – Logistic Regression ===")

lr_param_grid = {
    'C'      : [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver' : ['liblinear'],
}
lr_gs = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    lr_param_grid,
    cv=5, scoring='f1', n_jobs=-1
)
lr_gs.fit(X_tr_sm, y_tr_sm)

print(f"Best parameters : {lr_gs.best_params_}")
print(f"Best CV F1-score: {lr_gs.best_score_:.4f}")

lr_best = lr_gs.best_estimator_


=== b) Grid Search – Logistic Regression ===


Best parameters : {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Best CV F1-score: 0.9457


In [ ]:
# ── d) Overfitting check – Learning Curves ───────────────────────────────
print("=== d) Overfitting Check – Logistic Regression ===")

train_sz_lr, tr_scores_lr, val_scores_lr = learning_curve(
    lr_best, X_tr_sm, y_tr_sm,
    cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy'
)
tr_mean_lr  = tr_scores_lr.mean(axis=1)
val_mean_lr = val_scores_lr.mean(axis=1)
gap_lr      = tr_mean_lr[-1] - val_mean_lr[-1]

print(f"Final train accuracy     : {tr_mean_lr[-1]:.4f}")
print(f"Final validation accuracy: {val_mean_lr[-1]:.4f}")
print(f"Train–Val gap            : {gap_lr:.4f}  "
      f"({'possible overfit' if gap_lr > 0.05 else ' no significant overfit'})")


=== d) Overfitting Check – Logistic Regression ===


Final train accuracy     : 0.9505
Final validation accuracy: 0.9459
Train–Val gap            : 0.0045  (✅ no significant overfit)


In [15]:
# ── Predictions ──────────────────────────────────────────────────────────
y_pred_lr = lr_best.predict(X_test)
y_prob_lr = lr_best.predict_proba(X_test)[:, 1]

print("=== Logistic Regression – Classification Report ===")
print(classification_report(y_test, y_pred_lr, target_names=['Poor', 'Good']))


=== Logistic Regression – Classification Report ===
              precision    recall  f1-score   support

        Poor       0.94      0.89      0.91       149
        Good       0.91      0.95      0.93       171

    accuracy                           0.92       320
   macro avg       0.92      0.92      0.92       320
weighted avg       0.92      0.92      0.92       320



## 5 & 6. Classifier 2 – Support Vector Machine (SVM)

### c) Class Imbalance: `class_weight='balanced'`
The `class_weight='balanced'` parameter adjusts the SVM penalty inversely proportional to class frequencies, giving the minority class higher weight without generating synthetic samples.


In [16]:
# ── a) K-Fold Cross-Validation (k = 5) ───────────────────────────────────
print("=== a) 5-Fold Cross-Validation – SVM ===")

svm_base = SVC(kernel='rbf', random_state=42,
               class_weight='balanced', probability=True)
cv_scores_svm = cross_val_score(svm_base, X_train, y_train,
                                 cv=kfold, scoring='accuracy')
print(f"CV fold scores : {np.round(cv_scores_svm, 4)}")
print(f"Mean accuracy  : {cv_scores_svm.mean():.4f}  "
      f"(±{cv_scores_svm.std()*2:.4f})")


=== a) 5-Fold Cross-Validation – SVM ===


CV fold scores : [0.9453 0.9141 0.9141 0.9336 0.9412]
Mean accuracy  : 0.9296  (±0.0265)


In [17]:
# ── b) Grid Search Hyperparameter Tuning ─────────────────────────────────
print("=== b) Grid Search – SVM ===")

svm_param_grid = {
    'C'     : [0.1, 1, 10],
    'gamma' : ['scale', 'auto'],
    'kernel': ['rbf', 'linear'],
}
svm_gs = GridSearchCV(
    SVC(random_state=42, class_weight='balanced', probability=True),
    svm_param_grid,
    cv=5, scoring='f1', n_jobs=-1
)
svm_gs.fit(X_train, y_train)

print(f"Best parameters : {svm_gs.best_params_}")
print(f"Best CV F1-score: {svm_gs.best_score_:.4f}")

svm_best = svm_gs.best_estimator_


=== b) Grid Search – SVM ===


Best parameters : {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
Best CV F1-score: 0.9467


In [ ]:
# ── d) Overfitting check – Learning Curves ───────────────────────────────
print("=== d) Overfitting Check – SVM ===")

train_sz_svm, tr_scores_svm, val_scores_svm = learning_curve(
    svm_best, X_train, y_train,
    cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy'
)
tr_mean_svm  = tr_scores_svm.mean(axis=1)
val_mean_svm = val_scores_svm.mean(axis=1)
gap_svm      = tr_mean_svm[-1] - val_mean_svm[-1]

print(f"Final train accuracy     : {tr_mean_svm[-1]:.4f}")
print(f"Final validation accuracy: {val_mean_svm[-1]:.4f}")
print(f"Train–Val gap            : {gap_svm:.4f}  "
      f"({'possible overfit' if gap_svm > 0.05 else 'no significant overfit'})")


=== d) Overfitting Check – SVM ===


Final train accuracy     : 0.9505
Final validation accuracy: 0.9437
Train–Val gap            : 0.0068  (✅ no significant overfit)


In [19]:
# ── Predictions ──────────────────────────────────────────────────────────
y_pred_svm = svm_best.predict(X_test)
y_prob_svm = svm_best.predict_proba(X_test)[:, 1]

print("=== SVM – Classification Report ===")
print(classification_report(y_test, y_pred_svm, target_names=['Poor', 'Good']))


=== SVM – Classification Report ===
              precision    recall  f1-score   support

        Poor       0.94      0.90      0.92       149
        Good       0.92      0.95      0.93       171

    accuracy                           0.93       320
   macro avg       0.93      0.93      0.93       320
weighted avg       0.93      0.93      0.93       320



## 7 & 8. Classification Results (Quantitative & Visual) + Confusion Matrices

In [20]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# ── Row 0: Logistic Regression ────────────────────────────────────────────
# Confusion matrix
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lr),
                       display_labels=['Poor', 'Good']).plot(
    ax=axes[0, 0], colorbar=False, cmap='Blues')
axes[0, 0].set_title('Logistic Regression\nConfusion Matrix')

# ROC curve
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
auc_lr = auc(fpr_lr, tpr_lr)
axes[0, 1].plot(fpr_lr, tpr_lr, color='darkorange', lw=2,
                label=f'AUC = {auc_lr:.3f}')
axes[0, 1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0, 1].set(xlabel='False Positive Rate', ylabel='True Positive Rate',
               title='Logistic Regression\nROC Curve')
axes[0, 1].legend(loc='lower right')

# Learning curves
axes[0, 2].plot(train_sz_lr, tr_mean_lr,  'o-', color='tomato',  label='Train')
axes[0, 2].plot(train_sz_lr, val_mean_lr, 'o-', color='seagreen', label='Validation')
axes[0, 2].set(xlabel='Training Samples', ylabel='Accuracy',
               title='Logistic Regression\nLearning Curves')
axes[0, 2].legend(); axes[0, 2].grid(True, alpha=0.3)

# ── Row 1: SVM ────────────────────────────────────────────────────────────
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_svm),
                       display_labels=['Poor', 'Good']).plot(
    ax=axes[1, 0], colorbar=False, cmap='Greens')
axes[1, 0].set_title('SVM\nConfusion Matrix')

fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)
auc_svm = auc(fpr_svm, tpr_svm)
axes[1, 1].plot(fpr_svm, tpr_svm, color='darkorange', lw=2,
                label=f'AUC = {auc_svm:.3f}')
axes[1, 1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1, 1].set(xlabel='False Positive Rate', ylabel='True Positive Rate',
               title='SVM\nROC Curve')
axes[1, 1].legend(loc='lower right')

axes[1, 2].plot(train_sz_svm, tr_mean_svm,  'o-', color='tomato',  label='Train')
axes[1, 2].plot(train_sz_svm, val_mean_svm, 'o-', color='seagreen', label='Validation')
axes[1, 2].set(xlabel='Training Samples', ylabel='Accuracy',
               title='SVM\nLearning Curves')
axes[1, 2].legend(); axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('classification_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved → classification_results.png")


Plot saved → classification_results.png


## 9. Precision, Recall, and F-Measure

In [21]:
# ── Per-class and macro metrics ──────────────────────────────────────────
def metrics_table(y_true, y_pred, name):
    p  = precision_score(y_true, y_pred, average=None)
    r  = recall_score   (y_true, y_pred, average=None)
    f  = f1_score       (y_true, y_pred, average=None)
    pm = precision_score(y_true, y_pred, average='macro')
    rm = recall_score   (y_true, y_pred, average='macro')
    fm = f1_score       (y_true, y_pred, average='macro')
    acc = accuracy_score(y_true, y_pred)
    tbl = pd.DataFrame({
        'Class'    : ['Poor (0)', 'Good (1)', '── Macro ──'],
        'Precision': [f'{p[0]:.4f}', f'{p[1]:.4f}', f'{pm:.4f}'],
        'Recall'   : [f'{r[0]:.4f}', f'{r[1]:.4f}', f'{rm:.4f}'],
        'F-Measure': [f'{f[0]:.4f}', f'{f[1]:.4f}', f'{fm:.4f}'],
    })
    print(f"\n{'='*50}")
    print(f"  {name}  (Accuracy = {acc:.4f})")
    print('='*50)
    print(tbl.to_string(index=False))
    return pm, rm, fm, acc

pm_lr, rm_lr, fm_lr, acc_lr   = metrics_table(y_test, y_pred_lr, "Logistic Regression")
pm_svm, rm_svm, fm_svm, acc_svm = metrics_table(y_test, y_pred_svm, "SVM")



  Logistic Regression  (Accuracy = 0.9219)
      Class Precision Recall F-Measure
   Poor (0)    0.9429 0.8859    0.9135
   Good (1)    0.9056 0.9532    0.9288
── Macro ──    0.9242 0.9196    0.9211

  SVM  (Accuracy = 0.9281)
      Class Precision Recall F-Measure
   Poor (0)    0.9437 0.8993    0.9210
   Good (1)    0.9157 0.9532    0.9341
── Macro ──    0.9297 0.9263    0.9275


In [22]:
# ── Comparison bar chart ─────────────────────────────────────────────────
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
lr_vals  = [acc_lr,  pm_lr,  rm_lr,  fm_lr,  auc_lr]
svm_vals = [acc_svm, pm_svm, rm_svm, fm_svm, auc_svm]

x = np.arange(len(metrics_names))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, lr_vals,  w, label='Logistic Regression', color='steelblue')
b2 = ax.bar(x + w/2, svm_vals, w, label='SVM',                  color='coral')

for bar in [*b1, *b2]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set(xticks=x, xticklabels=metrics_names, ylim=(0, 1.12),
       ylabel='Score', title='Classifier Comparison – Key Metrics')
ax.legend()
plt.tight_layout()
plt.savefig('comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved → comparison_chart.png")

# ── Final summary ─────────────────────────────────────────────────────────
print("\n" + "="*55)
print(f"{'Metric':<22} {'Log. Regression':>16} {'SVM':>12}")
print("-"*55)
for name, lr_v, svm_v in zip(metrics_names, lr_vals, svm_vals):
    print(f"{name:<22} {lr_v:>16.4f} {svm_v:>12.4f}")
print("="*55)


Plot saved → comparison_chart.png

Metric                  Log. Regression          SVM
-------------------------------------------------------
Accuracy                         0.9219       0.9281
Precision                        0.9242       0.9297
Recall                           0.9196       0.9263
F1-Score                         0.9211       0.9275
ROC-AUC                          0.9810       0.9817
